# Fine-Tuning Gemma for Stock Market Analysis
This notebook fine-tunes Gemma using the Kaggle Stock Market Dataset.

**Before running:**
1. Download the dataset from https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset
2. Extract it so you have a folder of CSV files (one per ticker)
3. Set `DATASET_PATH` below to point to that folder
4. You need a HuggingFace account + token (free) from https://huggingface.co/settings/tokens
5. You need to accept Gemma's license at https://huggingface.co/google/gemma-2b

**Recommended:** Run this on Google Colab (free T4 GPU) or Kaggle Notebooks (free GPU)

In [ ]:
# Install dependencies
%pip install -q transformers datasets peft accelerate bitsandbytes trl pandas torch

In [ ]:
import os
import pandas as pd
import torch
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────

DATASET_PATH = "./stocks"          # folder containing CSV files from Kaggle
MODEL_NAME   = "google/gemma-2b"   # lighter than 7B, fits on free Colab GPU
OUTPUT_DIR   = "./gemma-stock-ft"  # where the fine-tuned model will be saved
HF_TOKEN     = "your_hf_token"     # from https://huggingface.co/settings/tokens

# How many CSV files to use (use None to use all ~7000, but that's slow)
MAX_TICKERS  = 50

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1 — Convert CSV Data into Q&A Training Examples
LLMs learn from text. We convert each row/window of stock data into
a question-answer pair that teaches the model to reason about stock movements.

In [ ]:
def row_to_qa(ticker: str, row: pd.Series, prev_row: pd.Series = None) -> dict:
    """Convert a single day's stock data into a Q&A training example."""
    date = str(row['Date'])[:10] if 'Date' in row else "unknown date"
    change = row['Close'] - row['Open']
    direction = "increased" if change >= 0 else "decreased"
    pct = abs(change / row['Open'] * 100) if row['Open'] != 0 else 0

    question = (
        f"On {date}, {ticker} opened at ${row['Open']:.2f}. "
        f"The high was ${row['High']:.2f}, the low was ${row['Low']:.2f}, "
        f"and volume was {int(row['Volume']):,}. "
        f"What was the closing price and how did the stock perform?"
    )

    answer = (
        f"{ticker} closed at ${row['Close']:.2f} on {date}, "
        f"{direction} by {pct:.2f}% from the opening price. "
        f"The trading range was ${row['Low']:.2f} to ${row['High']:.2f} "
        f"with a volume of {int(row['Volume']):,} shares traded."
    )

    # Gemma instruction format
    text = f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>"
    return {"text": text}


def window_to_qa(ticker: str, window: pd.DataFrame) -> dict:
    """Convert a 5-day window into a trend analysis Q&A."""
    start = str(window.iloc[0]['Date'])[:10] if 'Date' in window.columns else "start"
    end   = str(window.iloc[-1]['Date'])[:10] if 'Date' in window.columns else "end"
    start_price = window.iloc[0]['Open']
    end_price   = window.iloc[-1]['Close']
    total_change = end_price - start_price
    direction = "gained" if total_change >= 0 else "lost"
    pct = abs(total_change / start_price * 100) if start_price != 0 else 0
    avg_volume = window['Volume'].mean()

    question = (
        f"Analyze {ticker}'s stock performance from {start} to {end}. "
        f"The stock opened the period at ${start_price:.2f}."
    )

    answer = (
        f"Over this 5-day period, {ticker} {direction} {pct:.2f}%, "
        f"moving from ${start_price:.2f} to ${end_price:.2f}. "
        f"Average daily volume was {int(avg_volume):,} shares. "
        f"The overall trend was {'bullish' if total_change >= 0 else 'bearish'}."
    )

    text = f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>"
    return {"text": text}

In [ ]:
def build_dataset(dataset_path: str, max_tickers: int = None) -> list:
    csv_files = list(Path(dataset_path).glob("*.csv"))
    if max_tickers:
        csv_files = csv_files[:max_tickers]

    print(f"Processing {len(csv_files)} tickers...")
    examples = []

    for csv_file in csv_files:
        ticker = csv_file.stem.upper()
        try:
            df = pd.read_csv(csv_file)
            df = df.dropna(subset=['Open', 'Close', 'High', 'Low', 'Volume'])
            if len(df) < 10:
                continue

            # Single-day Q&A examples (sample every 5th row to keep dataset manageable)
            for _, row in df.iloc[::5].iterrows():
                examples.append(row_to_qa(ticker, row))

            # 5-day window trend examples
            for i in range(0, len(df) - 5, 10):
                window = df.iloc[i:i+5]
                examples.append(window_to_qa(ticker, window))

        except Exception as e:
            print(f"[WARNING] Skipping {ticker}: {e}")

    print(f"Built {len(examples)} training examples from {len(csv_files)} tickers.")
    return examples


examples = build_dataset(DATASET_PATH, max_tickers=MAX_TICKERS)

# Preview a few examples
for ex in examples[:2]:
    print(ex['text'])
    print("---")

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(examples)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(dataset['train'])} examples")
print(f"Test:  {len(dataset['test'])} examples")

## Step 2 — Load Gemma with 4-bit Quantization
We use QLoRA (4-bit quantization + LoRA adapters) so the model fits in GPU memory
and trains fast even on a free Colab T4 GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    add_eos_token=True
)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)
print("Model loaded.")

In [ ]:
# LoRA config — we only train a small set of adapter weights, not the full model
lora_config = LoraConfig(
    r=16,                          # rank — higher = more capacity but more memory
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1% of total params — that's the point of LoRA

## Step 3 — Fine-Tune

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

print("Starting fine-tuning...")
trainer.train()

## Step 4 — Save the Model

In [ ]:
# Save LoRA adapter weights
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## Step 5 — Quick Test

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def test_model(prompt: str):
    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the model's answer
    if "model" in response:
        response = response.split("model")[-1].strip()
    print(response)

test_model("On 2020-01-15, AAPL opened at $156.00. The high was $158.50, the low was $155.20, and volume was 80,000,000. What was the closing price and how did the stock perform?")

## Step 6 — Export to GGUF for LM Studio

To use your fine-tuned model in LM Studio, you need to convert it to GGUF format.
Run these commands in your terminal after training completes:

```bash
# 1. Merge LoRA weights into the base model
python merge_lora.py

# 2. Clone llama.cpp and convert
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt
python convert_hf_to_gguf.py ../gemma-stock-ft-merged --outfile ../gemma-stock-ft.gguf
```

Then in LM Studio: **My Models → Add Model → Load from path** and point to the `.gguf` file.

In [ ]:
# merge_lora.py content — run this separately after training
merge_script = '''
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL  = "google/gemma-2b"
LORA_PATH   = "./gemma-stock-ft"
MERGED_PATH = "./gemma-stock-ft-merged"
HF_TOKEN    = "your_hf_token"

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, token=HF_TOKEN, torch_dtype=torch.float16, device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

print("Merging LoRA weights...")
model = PeftModel.from_pretrained(base, LORA_PATH)
model = model.merge_and_unload()

print(f"Saving merged model to {MERGED_PATH}...")
model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print("Done.")
'''

with open("merge_lora.py", "w") as f:
    f.write(merge_script)
print("merge_lora.py saved.")